# 04 — RSNA preprocessing

Objectif : tester le chargement et le prétraitement d'images RSNA Pneumonia pour le pipeline VLM.

Le notebook fonctionne en deux modes :
- **RSNA réel** si les fichiers DICOM sont présents localement dans `data/raw/rsna/` ;
- **fallback synthétique** avec `data/sample_images/` si RSNA n'est pas encore téléchargé.

⚠️ Ce projet est un prototype pédagogique. Il n'est pas destiné au diagnostic médical.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

sys.path.append(str(Path('..').resolve()))

from src.preprocessing import (
    get_demo_image_paths,
    preprocess_for_vlm,
    find_rsna_images,
    find_image_files,
)


## 1. Configuration des chemins

In [ ]:
PROJECT_ROOT = Path('..').resolve()
RSNA_ROOT = PROJECT_ROOT / 'data' / 'raw' / 'rsna'
SYNTHETIC_ROOT = PROJECT_ROOT / 'data' / 'sample_images'

print('Project root:', PROJECT_ROOT)
print('RSNA root:', RSNA_ROOT)
print('Synthetic root:', SYNTHETIC_ROOT)


## 2. Vérification de la présence du dataset RSNA

In [ ]:
rsna_images = find_rsna_images(RSNA_ROOT)
synthetic_images = find_image_files(SYNTHETIC_ROOT)

print(f'Images RSNA trouvées: {len(rsna_images)}')
print(f'Images synthétiques trouvées: {len(synthetic_images)}')

if not rsna_images:
    print('RSNA absent localement : le notebook utilisera les images synthétiques comme fallback.')
else:
    print('RSNA détecté localement : le notebook utilisera les fichiers DICOM.')


## 3. Sélection automatique des images de démonstration

In [ ]:
demo_paths, source = get_demo_image_paths(
    rsna_root=RSNA_ROOT,
    synthetic_root=SYNTHETIC_ROOT,
    limit=5,
)

print('Source utilisée:', source)
for path in demo_paths:
    print('-', path)

assert demo_paths, 'Aucune image disponible. Vérifier data/sample_images/ ou télécharger RSNA.'


## 4. Chargement et prétraitement d'une image

In [ ]:
image_path = demo_paths[0]

image_pil = preprocess_for_vlm(image_path, size=(512, 512), output='pil')
image_np = preprocess_for_vlm(image_path, size=(512, 512), output='numpy')

print('Image:', image_path.name)
print('PIL mode:', image_pil.mode)
print('PIL size:', image_pil.size)
print('Numpy shape:', image_np.shape)
print('Numpy range:', float(image_np.min()), '→', float(image_np.max()))


## 5. Affichage de l'image prétraitée

In [ ]:
plt.figure(figsize=(5, 5))
plt.imshow(image_pil)
plt.title(f'{source.upper()} — {image_path.name}')
plt.axis('off')
plt.show()


## 6. Mini-test sur plusieurs images

In [ ]:
results = []

for path in demo_paths:
    try:
        arr = preprocess_for_vlm(path, size=(512, 512), output='numpy')
        results.append({
            'path': str(path),
            'shape': arr.shape,
            'min': float(arr.min()),
            'max': float(arr.max()),
            'status': 'ok',
        })
    except Exception as exc:
        results.append({
            'path': str(path),
            'shape': None,
            'min': None,
            'max': None,
            'status': f'error: {exc}',
        })

results


## 7. Conclusion

Le pipeline permet maintenant de :

- charger des images synthétiques déjà présentes dans le repo ;
- charger des fichiers DICOM RSNA si le dataset réel est disponible localement ;
- convertir les images en RGB ;
- redimensionner en 512×512 ;
- fournir soit une image PIL pour un VLM, soit un array NumPy normalisé pour inspection.

Les données RSNA réelles doivent rester locales et ne doivent pas être commitées sur GitHub.